In [ ]:
import numpy as np
import pandas as pd
import joblib

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    roc_auc_score,
    roc_curve,
    classification_report
    )

from wordcloud import WordCloud, STOPWORDS

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.pipeline import Pipeline

import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

In [ ]:
df = pd.read_csv("./data.csv", encoding='ISO-8859-1')

In [ ]:
df.head()

In [ ]:
print("Number of rows are: ",df.shape[0])
print("Number of columns are: ",df.shape[1])

In [ ]:
df.info()

In [ ]:
dup = df.duplicated().sum()
print(f'number of duplicated rows are {dup}')

In [ ]:
df.isnull().sum()

In [ ]:
df.columns

In [ ]:
df.describe(include= 'all').round(2)

In [ ]:
for i in df.columns.tolist():
  print("No. of unique values in",i,"is",df[i].nunique())

In [ ]:
df.rename(columns={"v1": "Category", "v2": "Message"}, inplace=True)

In [ ]:
df.drop(columns={'Unnamed: 2','Unnamed: 3','Unnamed: 4'}, inplace=True)

In [ ]:
df['Spam'] = df['Category'].apply(lambda x: 1 if x == 'spam' else 0)

In [ ]:
df.head()

In [ ]:
spread = df['Category'].value_counts()
plt.rcParams['figure.figsize'] = (5,5)

spread.plot(kind = 'pie', autopct='%1.2f%%', cmap='Set1')
plt.title(f'Distribution of Spam vs Ham')

plt.show()

In [ ]:
df_spam = df[df['Category']=='spam'].copy()

In [ ]:
comment_words = ''

stopwords = set(STOPWORDS)

for val in df_spam.Message:

    val = str(val)

    tokens = val.split()

    for i in range(len(tokens)):
        tokens[i] = tokens[i].lower()

    comment_words += " ".join(tokens)+" "

wordcloud = WordCloud(width = 1000, height = 500,
                background_color ='white',
                stopwords = stopwords,
                min_font_size = 10,
                max_words = 1000,
                colormap = 'gist_heat_r').generate(comment_words)

plt.figure(figsize = (6,6), facecolor = None)
plt.title('Most Used Words In Spam Messages', fontsize = 15, pad=20)
plt.imshow(wordcloud)
plt.axis("off")
plt.tight_layout(pad = 0)

plt.show()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df.Message,
    df.Spam,
    test_size=0.25,
    stratify=df.Spam,
    random_state=42
    )

In [ ]:
def evaluate_model(model, X_train, X_test, y_train, y_test, verbose=True, show_plots=True):
    """Fit a model, evaluate on train/test, and return metric dict."""
    model.fit(X_train, y_train)

    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    pred_prob_train = model.predict_proba(X_train)[:, 1]
    pred_prob_test = model.predict_proba(X_test)[:, 1]

    roc_auc_train = roc_auc_score(y_train, pred_prob_train)
    roc_auc_test = roc_auc_score(y_test, pred_prob_test)

    if verbose:
        print("\nTrain ROC AUC:", roc_auc_train)
        print("Test ROC AUC:", roc_auc_test)

    if show_plots:
        fpr_train, tpr_train, _ = roc_curve(y_train, pred_prob_train)
        fpr_test, tpr_test, _ = roc_curve(y_test, pred_prob_test)
        plt.plot([0, 1], [0, 1], 'k--')
        plt.plot(fpr_train, tpr_train, label=f"Train ROC AUC: {roc_auc_train:.2f}")
        plt.plot(fpr_test, tpr_test, label=f"Test ROC AUC: {roc_auc_test:.2f}")
        plt.legend()
        plt.title("ROC Curve")
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.show()

    cm_train = confusion_matrix(y_train, y_pred_train)
    cm_test = confusion_matrix(y_test, y_pred_test)

    if show_plots:
        fig, ax = plt.subplots(1, 2, figsize=(11, 4))
        print("\nConfusion Matrix:")
        sns.heatmap(cm_train, annot=True, xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'], cmap="Oranges", fmt='.4g', ax=ax[0])
        ax[0].set_xlabel("Predicted Label")
        ax[0].set_ylabel("True Label")
        ax[0].set_title("Train Confusion Matrix")

        sns.heatmap(cm_test, annot=True, xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'], cmap="Oranges", fmt='.4g', ax=ax[1])
        ax[1].set_xlabel("Predicted Label")
        ax[1].set_ylabel("True Label")
        ax[1].set_title("Test Confusion Matrix")

        plt.tight_layout()
        plt.show()

    cr_train = classification_report(y_train, y_pred_train, output_dict=True, zero_division=0)
    cr_test = classification_report(y_test, y_pred_test, output_dict=True, zero_division=0)
    if verbose:
        print("\nTrain Classification Report:")
        print(pd.DataFrame(cr_train).T.to_markdown())

        print("\nTest Classification Report:")
        print(pd.DataFrame(cr_test).T.to_markdown())

    precision_train = cr_train['weighted avg']['precision']
    precision_test = cr_test['weighted avg']['precision']
    recall_train = cr_train['weighted avg']['recall']
    recall_test = cr_test['weighted avg']['recall']
    acc_train = accuracy_score(y_true=y_train, y_pred=y_pred_train)
    acc_test = accuracy_score(y_true=y_test, y_pred=y_pred_test)
    f1_train = cr_train['weighted avg']['f1-score']
    f1_test = cr_test['weighted avg']['f1-score']

    return {
        "precision": precision_test,
        "recall": recall_test,
        "accuracy": acc_test,
        "f1": f1_test,
        "roc_auc": roc_auc_test
    }

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("tfidf", TfidfVectorizer(stop_words="english")),
        ("clf", LogisticRegression(max_iter=1000))
    ]),
    "Random Forest": Pipeline([
        ("tfidf", TfidfVectorizer(stop_words="english")),
        ("clf", RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            n_jobs=-1
        ))
    ])
}

In [ ]:
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_rows = []
for name, model in models.items():
    scores = cross_validate(
        model,
        df.Message,
        df.Spam,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )
    cv_rows.append({
        "model": name,
        "accuracy": scores["test_accuracy"].mean(),
        "precision": scores["test_precision"].mean(),
        "recall": scores["test_recall"].mean(),
        "f1": scores["test_f1"].mean(),
        "roc_auc": scores["test_roc_auc"].mean()
    })

cv_df = pd.DataFrame(cv_rows).set_index("model").sort_values("roc_auc", ascending=False)
cv_df

test_rows = []
for name, model in models.items():
    metrics = evaluate_model(model, X_train, X_test, y_train, y_test, verbose=False, show_plots=False)
    test_rows.append({"model": name, **metrics})

test_df = pd.DataFrame(test_rows).set_index("model").sort_values("roc_auc", ascending=False)
test_df

best_model_name = test_df.index[0]
best_model = models[best_model_name]
print(f"Best model by holdout ROC-AUC: {best_model_name}")

_ = evaluate_model(best_model, X_train, X_test, y_train, y_test, verbose=True, show_plots=True)

final_model = models[best_model_name]
final_model.fit(df.Message, df.Spam)
joblib.dump(final_model, "spam_classifier.joblib")

In [ ]:
def detect_spam(email_text):
    prediction = final_model.predict([email_text])
    return "This is a Ham Email!" if prediction == 0 else "This is a Spam Email!"

In [ ]:
sample_email = "Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's"
result = detect_spam(sample_email)
print(result)